In [1]:
import torch
import torchvision
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from sklearn.metrics import f1_score as sklearn_f1_score
from sklearn.metrics import precision_score, recall_score, accuracy_score

import torch.nn as nn
import torch.optim as optim
import copy

import tqdm

import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold

import pandas as pd
import matplotlib.pyplot as plt

from foldrm import Classifier
from utils import split_data # Or your stratified version if you prefer
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

c:\Users\anmarch\repos\extinction-risk-inference\CONFOLD\foldrm.py:30: SyntaxWarning: invalid escape sequence '\#'
  print(f"\#\#\#\#\# Instructions \#\#\#\#\#")
c:\Users\anmarch\repos\extinction-risk-inference\CONFOLD\foldrm.py:31: SyntaxWarning: invalid escape sequence '\#'
  print(f"Manually defined rules should take the following form: \n (with confidence \#) class = 'label' if 'attribute name/index' 'symbol' 'value' ")


In [2]:
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU found")

GPU available: False
No GPU found


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
random.seed(42)

In [5]:
model, data = final_extinctionrisk_dataframe()

In [6]:
categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)
print(one_hot_dataset)

#for testing
#one_hot_dataset = data[model.numeric]

X = one_hot_dataset
#X = one_hot_dataset.convert_dtypes()

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)



      Beak_length_culmen  Beak_depth  Tarsus_length  Wing_length  \
0                   90.5        23.4          481.2        579.0   
1                   89.2        23.6          466.7        510.5   
2                   86.5        17.2          308.0        604.5   
3                   68.6        16.3          292.5        629.5   
4                   33.0         6.2           58.8        191.2   
...                  ...         ...            ...          ...   
9048                12.6         5.2           16.9         71.8   
9049                13.0         5.0           16.4         70.2   
9050                13.0         4.7           17.7         76.7   
9051                15.4         4.4           16.9         77.0   
9052                14.6         4.6           16.5         72.9   

      Hand_wing_index  Tail_length  Minimum_latitude  Maximum_latitude  \
0                 0.1        309.2            -30.91             22.47   
1                 0.1        291.7 

In [7]:
X.to(device)
y.to(device)

tensor([0., 1., 1.,  ..., 0., 0., 0.])

In [8]:
print(one_hot_dataset)

      Beak_length_culmen  Beak_depth  Tarsus_length  Wing_length  \
0                   90.5        23.4          481.2        579.0   
1                   89.2        23.6          466.7        510.5   
2                   86.5        17.2          308.0        604.5   
3                   68.6        16.3          292.5        629.5   
4                   33.0         6.2           58.8        191.2   
...                  ...         ...            ...          ...   
9048                12.6         5.2           16.9         71.8   
9049                13.0         5.0           16.4         70.2   
9050                13.0         4.7           17.7         76.7   
9051                15.4         4.4           16.9         77.0   
9052                14.6         4.6           16.5         72.9   

      Hand_wing_index  Tail_length  Minimum_latitude  Maximum_latitude  \
0                 0.1        309.2            -30.91             22.47   
1                 0.1        291.7 

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [10]:
print(np.shape(X))
print(np.shape(X)[1])

torch.Size([9053, 393])
393


In [11]:
class TwoLayer(torch.nn.Module):
    def __init__(self, input_size):
        super(TwoLayer, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 60)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(60, 60)
        self.linear3 = torch.nn.Linear(60, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.activation(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        return x

tinymodel = TwoLayer(np.shape(X)[1])

print(np.shape(X)[1])

print('The model:')
print(tinymodel)



393
The model:
TwoLayer(
  (linear1): Linear(in_features=393, out_features=60, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=60, out_features=60, bias=True)
  (linear3): Linear(in_features=60, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [12]:
tinymodel.to(device)

TwoLayer(
  (linear1): Linear(in_features=393, out_features=60, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=60, out_features=60, bias=True)
  (linear3): Linear(in_features=60, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [13]:
def model_train(model, X_train, y_train, X_val, y_val):
    # loss function and optimizer
    loss_fn = nn.BCELoss()  # binary cross entropy
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
 
    n_epochs = 100   # number of epochs to run
    batch_size = 10  # size of each batch
    batch_start = torch.arange(0, len(X_train), batch_size)
 
    # Hold the best model
    best_acc = - np.inf   # init to negative infinity
    best_rec = - np.inf   # init to negative infinity
    best_pre = - np.inf   # init to negative infinity
    best_f1 = - np.inf   # init to negative infinity
    best_weights = None
 
    for epoch in range(n_epochs):
        model.train()
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for start in bar:
                # take a batch
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size].unsqueeze(dim=1)
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
        # evaluate accuracy at end of each epoch
        model.eval()
        y_pred = model(X_val)
        preds_cpu = y_pred.cpu()
        labels_cpu = y_val.cpu()
        preds_numpy = np.round(preds_cpu.detach().numpy().flatten())
        labels_numpy = labels_cpu.detach().numpy()
        f1 = sklearn_f1_score(labels_numpy, preds_numpy, average=None)
        acc = accuracy_score(labels_numpy, preds_numpy)
        precision = precision_score(labels_numpy, preds_numpy, average=None)
        recall = recall_score(labels_numpy, preds_numpy, average=None)
        
        if acc > best_acc:
            best_acc = acc
            best_weights = copy.deepcopy(model.state_dict())
            best_rec = recall
            best_pre = precision
            best_f1 = f1
            stop_state = epoch
    # restore model and return best accuracy
    model.load_state_dict(best_weights)
    return best_acc, best_rec, best_pre, best_f1, stop_state

In [14]:
cv_scores_wide = []
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# create model, train, and get accuracy
model = tinymodel
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

C:\Users\anmarch\AppData\Local\Temp\ipykernel_54332\1378900292.py:36: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  loss=float(loss),
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

Accuracy:0.8652678078409718
Precision:[0.879375   0.75829384]
Recall[0.96502058 0.45325779]
F1:[0.92020929 0.56737589]
epoch:63


In [15]:
const_bnn_prior_parameters = {
        "prior_mu": 0.0,
        "prior_sigma": 1.0,
        "posterior_mu_init": 0.0,
        "posterior_rho_init": -3.0,
        "type": "Reparameterization",  # Flipout or Reparameterization
        "moped_enable": True,  # True to initialize mu/sigma from the pretrained dnn weights
        "moped_delta": 0.5,
}

dnn_to_bnn(tinymodel, const_bnn_prior_parameters)

print('The model:')
print(tinymodel)

print('\n\nJust one layer:')
print(tinymodel.linear2)

print('\n\nModel params:')
for param in tinymodel.parameters():
    print(param)

print('\n\nLayer params:')
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TwoLayer(
  (linear1): LinearReparameterization()
  (activation): ReLU()
  (linear2): LinearReparameterization()
  (linear3): LinearReparameterization()
  (sigmoid): Sigmoid()
)


Just one layer:
LinearReparameterization()


Model params:
Parameter containing:
tensor([[ 0.0140,  0.1154, -0.0272,  ..., -0.0326,  0.0377, -0.0650],
        [-0.1694, -0.2499, -0.1663,  ...,  0.0062, -0.1125, -0.1651],
        [ 0.0385,  0.1248,  0.0708,  ..., -0.0732,  0.2173,  0.0722],
        ...,
        [-0.1832, -0.1930, -0.2484,  ..., -0.0159, -0.0362, -0.2292],
        [-0.1063,  0.0030, -0.0381,  ...,  0.0079, -0.0194, -0.0546],
        [-0.0485, -0.1707, -0.0758,  ...,  0.0407, -0.0792, -0.0346]],
       requires_grad=True)
Parameter containing:
tensor([[-4.9583, -2.8238, -4.2915,  ..., -4.1087, -3.9629, -3.4105],
        [-2.4261, -2.0166, -2.4450,  ..., -5.7710, -2.8495, -2.4531],
        [-3.9399, -2.7430, -3.3233,  ..., -3.2893, -2.1648, -3.3029],
        ...,
        [-2.3440, -2.2

In [16]:
tinymodel.to(device)

TwoLayer(
  (linear1): LinearReparameterization()
  (activation): ReLU()
  (linear2): LinearReparameterization()
  (linear3): LinearReparameterization()
  (sigmoid): Sigmoid()
)

In [17]:
model = tinymodel
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

Accuracy:0.8586416344561016
Precision:[0.87236679 0.74619289]
Recall[0.96570645 0.41643059]
F1:[0.91666667 0.53454545]
epoch:92
